# Reproducing Windolf et al. (2025), Extended Data Fig. 3, from dandiset 001290

Paper: Windolf, Yu, Paulk et al. (2025), "DREDge: robust motion correction for high-density
extracellular recordings across species", Nature Methods, doi:10.1038/s41592-025-02614-5.

Dataset: DANDI:001290, "Neuropixels pulsation datasets", version 0.260827.1449
(doi:10.48324/dandi.001290/0.260827.1449). The paper cites the dandiset without a version (Data
availability). This is the only published version, so it is pinned; the draft has the same single
asset and size, and a draft can change.

Text: The claims file (`claims/001290/10.1038-s41592-025-02614-5.yaml`) was written from the
accepted author manuscript in PubMed Central (PMC13055889) with its Methods, legends and
supplement. Its opening comment defines "the pipeline" (steps 1 to 3) that the criteria refer to;
the helper module `pulsation.py` says where each parameter comes from.

| Claim | Tested here |
|---|---|
| edf3a-ap-dredge-slow-trend | tested (step 1 reading (ii) only; reading (i) not runnable) |
| edf3b-hdbscan-clusters | tested |
| edf3c-spline-subsecond | tested |
| edf3de-spline-improves-registration | tested |
| 30 claims with `data: elsewhere` (human, mouse, rat, simulated and other macaque data) | blocked: not attempted in this run |

Step 1 reading (i) of the claims file (the IBL preprocessing followed by DARTsort detection,
denoising and localization) was tried outside this notebook: DARTsort (github.com/cwindolf/dartsort
at commit 459cf656) on CPU did not finish its subtraction step on a 31 s segment within 13.5
minutes, which puts the 600 s recording above 4 hours, beyond the 60 minute limit of a notebook
and the time of this run. That trial is not run here and decides no outcome; it is why only
reading (ii) enters the outcomes.

In [ ]:
import os
for v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"]:
    os.environ[v] = "1"
import time
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import h5py
import matplotlib.pyplot as plt
import requests
import pulsation as P

API = f"https://api.dandiarchive.org/api/dandisets/{P.DANDISET_ID}/versions/{P.DANDISET_VERSION}/"
meta = requests.get(API).json()
assets = requests.get(API + "assets/").json()["results"]
print(meta["name"], "|", meta["doi"])
print(meta["description"])
print("subjects", meta["assetsSummary"]["numberOfSubjects"], "| species",
      [s["name"] for s in meta["assetsSummary"]["species"]])
for a in assets:
    print(a["path"], a["size"], "bytes")
URL = P.asset_url()

## Data check

The file is listed in full (every group and dataset, with shapes and the electrode table's columns),
then the quantities the analysis uses are checked against their description.

In [ ]:
f = P.open_file(URL)
def show(name, obj):
    if isinstance(obj, h5py.Dataset):
        if name.startswith("specifications"):
            return
        val = obj[()] if obj.shape == () else None
        print(f"{name}  {obj.shape} {obj.dtype}" + (f"  = {val!r}" if val is not None else ""))
f.visititems(show)
el = f["general/extracellular_ephys/electrodes"]
print("electrode table columns:", list(el.attrs["colnames"]))
for k in el.attrs["colnames"]:
    if k != "group":
        v = el[k][:]
        print(f"  {k}: {len(v)} rows, unique values {len(np.unique(v))}, first {v[:4]}")

In [ ]:
ap = f[P.AP_PATH]
lf = f[P.LF_PATH]
fs_ap = float(ap["starting_time"].attrs["rate"])
fs_lf = float(lf["starting_time"].attrs["rate"])
dur_ap = ap["data"].shape[0] / fs_ap
dur_lf = lf["data"].shape[0] / fs_lf
gain_uv = ap["data"].attrs["conversion"] * 1e6
locs, shift = P.probe_geometry(f)
print("AP:", ap["data"].shape, "at", fs_ap, "Hz,", round(dur_ap, 3), "s, starting time",
      ap["starting_time"][()], "s, gain", gain_uv, "uV per bit (NP 1.0 AP gain 500 gives 2.34375)")
print("LF:", lf["data"].shape, "at", fs_lf, "Hz,", round(dur_lf, 3), "s, starting time",
      lf["starting_time"][()], "s, gain", lf["data"].attrs["conversion"] * 1e6, "uV per bit")
print("AP and LF durations differ by", round(abs(dur_ap - dur_lf) * 1e3, 2), "ms")
print("AP electrode rows", ap["electrodes"][0], "to", ap["electrodes"][-1],
      "| LF rows", lf["electrodes"][0], "to", lf["electrodes"][-1])
print("channel x values (um):", np.unique(locs[:, 0]), "| y from", locs[:, 1].min(), "to",
      locs[:, 1].max(), "um, step", np.unique(np.diff(np.unique(locs[:, 1]))))
print("inter_sample_shift from", shift.min(), "to", shift.max(), "(13 ADC groups on NP 1.0)")
print("session:", f["session_description"][()].decode())
print("trials, units, stimulus or processing tables:", list(f["intervals"]) if "intervals" in f else "none",
      list(f["units"]) if "units" in f else "none", list(f["processing"]),
      list(f["stimulus/presentation"]))
YLO, YHI = float(locs[:, 1].min()), float(locs[:, 1].max())
T_END = dur_ap

## Spike detection and localization (pipeline step 1, reading (ii))

Every AP block of 78,125 samples (the file's chunk length) is streamed, converted to microvolts,
bandpass filtered (300 to 6,000 Hz), median referenced, and passed to SpikeInterface's
`detect_peaks` (locally exclusive, threshold 6 times the noise level, both signs) and
`localize_peaks` (monopolar triangulation, 75 um radius), with 3,000 samples of each neighbouring
block as margin. This takes about 15 minutes.

In [ ]:
t0 = time.time()
noise = P.noise_levels(URL)
sp = P.detect_spikes(URL, noise=noise, n_workers=4)
print(f"detection and localization took {(time.time() - t0) / 60:.1f} min")

In [ ]:
n = len(sp["t"])
print("noise level (uV, sd from MAD) median", round(float(np.median(noise)), 2), "range",
      round(float(noise.min()), 2), "to", round(float(noise.max()), 2))
print("spikes detected:", n, "| per second", round(n / T_END, 1))
print("time from", round(sp["t"].min(), 3), "to", round(sp["t"].max(), 3), "s")
for k, unit in [("amp", "uV"), ("x", "um"), ("y", "um"), ("z", "um")]:
    q = np.percentile(sp[k], [0, 1, 50, 99, 100])
    print(f"{k} ({unit}): min {q[0]:.1f}, 1% {q[1]:.1f}, median {q[2]:.1f}, 99% {q[3]:.1f}, max {q[4]:.1f}")
out = (sp["y"] < YLO) | (sp["y"] > YHI)
print("spikes localized beyond the probe's ends:", int(out.sum()), f"({out.mean():.1%})")
cnt, _ = np.histogram(sp["t"], bins=np.arange(0, T_END + 10, 10))
print("spikes per 10 s: min", cnt.min(), "median", int(np.median(cnt)), "max", cnt.max())
hd, he = np.histogram(sp["y"], bins=np.arange(-200, 4001, 400))
print("spikes per 400 um of depth:", dict(zip(he[:-1].astype(int), hd)))

The checks against the description: one subject (Macaca nemestrina, as the archive says), one
Neuropixels 1.0 probe of 384 channels in the standard staggered layout at 20 um vertical spacing, an AP
band at 30 kHz with the NP 1.0 gain, and an LF band at 2.5 kHz. The AP and LF series share a start
time of 0 and have the same duration, which is consistent with one clock but does not prove it.
The file has no trial, stimulus or unit tables, so no contrast between conditions can test the
clock; none of the analyses here uses one. The session description names static gratings at 8
orientations, where the paper's Methods say most stimuli were binary or ternary noise at 40 Hz;
no stimulus record is in the file to tell which. The dandiset holds one file and one recording, so
no recording can be deposited twice.

## edf3a-ap-dredge-slow-trend

> In the fast-motion macaque recording, DREDge's spike-based (AP) motion estimate follows the drift
> visible in the unregistered spike raster.

Criterion: mean template correlation after DREDge correction against the unregistered raster, under
registration readings (a) nonrigid and (b) rigid, and step 1 readings (i) and (ii). Only (ii) is
run. The code's default for raster smoothing (`gaussian_smoothing_sigma_um` and `_s` left as None)
is run as the variant the claims file names.

In [ ]:
reg = {}
for name, rigid in [("a_nonrigid", False), ("b_rigid", True)]:
    t0 = time.time()
    reg[name] = P.register(sp, rigid=rigid, smooth=True)
    print(name, "registered in", round(time.time() - t0, 1), "s")
me_default = P.register(sp, rigid=False, smooth=False)
print("code default (smoothing arguments None) against explicit 1 um, 1 s smoothing, nonrigid:"
      " largest difference in displacement", float(np.abs(me_default.displacement - reg["a_nonrigid"].displacement).max()), "um")

In [ ]:
lo0, hi0 = P.probe_extent(None, YLO, YHI)
tc_unreg, corr_unreg = P.template_correlation(sp["t"], sp["y"], sp["amp"], lo0, hi0, t_max=T_END)
print(f"unregistered: mean template correlation {tc_unreg:.4f} over {np.isfinite(corr_unreg).sum()} time bins")
yreg = {}
tc_reg = {}
for name, me in reg.items():
    yreg[name] = me.correct_s(sp["t"], sp["y"])
    lo, hi = P.probe_extent(me, YLO, YHI)
    tc, corr = P.template_correlation(sp["t"], yreg[name], sp["amp"], lo, hi, t_max=T_END)
    tc_reg[name] = tc
    d = me.displacement
    dd = np.abs(np.diff(d, axis=-1))
    better = np.nanmean(corr > corr_unreg)
    print(f"{name}: mean template correlation {tc:.4f} (change {tc - tc_unreg:+.4f}); time bins"
          f" improved {better:.1%}")
    print(f"   displacement range {d.min():.1f} to {d.max():.1f} um, sd {d.std():.1f} um; "
          f"change between 1 s bins median {np.median(dd):.1f}, 99th percentile {np.percentile(dd, 99):.1f} um")
    if d.ndim == 2:
        print("   nonrigid window centres (um):", np.round(me.spatial_bin_centers_um).astype(int))
        print("   sd of displacement per window (um):", np.round(d.std(axis=1), 1))
# slow trend: 60 s means of the rigid trace
dr = reg["b_rigid"].displacement
print("rigid displacement, mean per 60 s (um):", np.round([dr[i:i + 60].mean() for i in range(0, len(dr), 60)], 1))
print("rigid displacement, sd within 60 s (um):", np.round([dr[i:i + 60].std() for i in range(0, len(dr), 60)], 1))
lo, hi = P.probe_extent(reg["a_nonrigid"], YLO, YHI)
tc_def, _ = P.template_correlation(sp["t"], me_default.correct_s(sp["t"], sp["y"]), sp["amp"], lo, hi, t_max=T_END)
print(f"variant, code default smoothing, nonrigid: {tc_def:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (name, me) in zip(axes, reg.items()):
    sel = sp["amp"] > np.quantile(sp["amp"], 0.5)
    ax.scatter(sp["t"][sel], sp["y"][sel], s=0.2, c=np.log(sp["amp"][sel]), cmap="Greys", rasterized=True)
    if me.displacement.ndim == 2:
        for j, c in enumerate(me.spatial_bin_centers_um):
            ax.plot(me.time_bin_centers_s, c + me.displacement[j], "r", lw=0.6)
    else:
        for c in (1000, 2000, 3000):
            ax.plot(me.time_bin_centers_s, c + me.displacement, "r", lw=0.6)
    ax.set_title(f"unregistered spikes (top 50% amplitude), {name} estimate in red")
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("depth (um)")
axes[0].set_ylim(-200, 4000)
print("spikes beyond the plotted depth range (-200 to 4000 um):",
      int(((sp["y"] < -200) | (sp["y"] > 4000)).sum()))
plt.show()

**Result: reproduced, resting on the assumption that reading (ii) of step 1 stands for step 1, as reading (i) could not be run.** The mean template correlation rises from 0.3023 unregistered to 0.3319 with nonrigid registration (a) and 0.3338 with rigid registration (b), and is higher in 70.8% and 67.5% of the 1 s bins. The estimate is large (rigid range -84.1 to 131.5 um, s.d. 40.8 um) and moves between 1 s bins by a median of 3.3 um (nonrigid) and 7.6 um (rigid). Its 60 s means go from -33.7 to 38.2 um, while its s.d. within each 60 s is 21.1 to 45.4 um, so most of what it follows is motion on the scale of seconds, with a slow trend under it. Leaving the smoothing arguments at the code's default gives the same displacement (largest difference 0.0 um).

## Fast motion: clustering and splines (pipeline step 3)

All combinations of the open readings: registration (a) or (b); amplitude cut top 10% or top 25%;
HDBSCAN features (lateral position, registered depth, amplitude) or (registered depth, amplitude),
z-scored; `min_cluster_size` 25 or 100; spline least squares or penalized. One cell computes them;
the claims below read their parts of the table.

In [ ]:
rng_c = np.random.default_rng(20251)   # edf3c section: scrambled controls
N_PERM = 100
combos = []
t0 = time.time()
for regname in reg:
    me = reg[regname]
    yr = yreg[regname]
    for frac in (0.10, 0.25):
        sel = P.select_high_amplitude(sp["amp"], frac)
        for feats in ("x,y,amp", "y,amp"):
            F = (np.c_[sp["x"][sel], yr[sel], sp["amp"][sel]] if feats == "x,y,amp"
                 else np.c_[yr[sel], sp["amp"][sel]])
            for mcs in (25, 100):
                labels = P.cluster_spikes(F, mcs)
                keep, cen, lab = P.center_clusters(yr[sel], labels)
                tt = sp["t"][sel][keep]
                sizes = np.bincount(labels[labels >= 0])
                for pen in (False, True):
                    spline, info = P.fit_spline(tt, cen, 0.0, T_END, penalized=pen)
                    combos.append(dict(reg=regname, frac=frac, feats=feats, mcs=mcs, pen=pen,
                                       n_sel=int(sel.sum()), labels=labels, sizes=sizes, keep=keep,
                                       cen=cen, lab=lab, tt=tt, spline=spline, info=info))
print(len(combos), "combinations in", round((time.time() - t0) / 60, 1), "min")

## edf3b-hdbscan-clusters

> After AP DREDge registration, high-amplitude spikes form clusters that HDBSCAN isolates using
> their registered positions and amplitudes.

Holds when HDBSCAN returns at least two clusters other than noise, in every combination of amplitude
cut, features, `min_cluster_size` and registration reading.

In [ ]:
print(f"{'reg':11s} {'cut':>4s} {'features':9s} {'mcs':>4s} {'spikes':>7s} {'clusters':>8s} {'noise':>6s} {'outliers':>8s}  sizes (largest first)")
n_ok = 0
seen = set()
for c in combos:
    key = (c["reg"], c["frac"], c["feats"], c["mcs"])
    if key in seen:
        continue
    seen.add(key)
    k = len(c["sizes"])
    n_ok += k >= 2
    noise_frac = np.mean(c["labels"] < 0)
    n_out = int((c["labels"] >= 0).sum() - c["keep"].sum())
    print(f"{c['reg']:11s} {c['frac']:4.2f} {c['feats']:9s} {c['mcs']:4d} {c['n_sel']:7d} {k:8d} {noise_frac:6.1%} {n_out:8d}  {sorted(c['sizes'].tolist(), reverse=True)[:12]}")
print(f"combinations with at least two clusters: {n_ok} of {len(seen)}")

In [ ]:
# spread of the clusters in registered depth (a cluster of one unit spans tens of um)
for c in combos:
    if c["pen"] or c["frac"] != 0.10 or c["mcs"] != 100:
        continue
    sds = [np.std(c["cen"][c["lab"] == j]) for j in np.unique(c["lab"])]
    print(f"{c['reg']}, top 10%, {c['feats']}, mcs 100: within-cluster sd of registered depth (um)",
          np.round(sorted(sds), 1))

**Result: reproduced.** All 16 combinations return at least two clusters other than noise (6 to 41 clusters), as the criterion requires. The clusters are broad: with the top 10% cut and min_cluster_size 100 their s.d. of registered depth is 21.6 to 134.8 um, and a cluster of about 16,000 spikes appears in every combination, so they are not single units.

## edf3c-spline-subsecond

> A smoothing spline fit to the centered positions of the clustered spikes captures sub-second
> displacement left after AP DREDge registration.

For each combination, the pooled within-cluster sd of corrected depth after spline correction is
compared with the same spline trace with its 0.4 s bins randomly permuted (100 permutations, fixed
seed). Reproduced when the spline value is below the 5th percentile of the scrambled values in every
combination.

In [ ]:
grid = np.arange(0.0, T_END, 0.01)
freqs = np.fft.rfftfreq(len(grid), 0.01)
print(f"{'reg':11s} {'cut':>4s} {'features':9s} {'mcs':>4s} {'spline':9s} {'sd none':>8s} {'sd spline':>9s} {'scr 5%':>7s} {'scr med':>7s} {'pass':>5s} {'range um':>17s} {'sd um':>6s} {'>0.5Hz':>7s} {'edf':>6s}")
n_pass = 0
for c in combos:
    tt, cen, lab = c["tt"], c["cen"], c["lab"]
    s_none = P.pooled_within_sd(cen, lab)
    s_spl = P.pooled_within_sd(cen - c["spline"](tt), lab)
    scr = np.array([P.pooled_within_sd(cen - tr, lab)
                    for tr in P.scrambled_traces(c["spline"], tt, 0.0, T_END, N_PERM, rng_c)])
    p5 = np.percentile(scr, 5)
    ok = s_spl < p5
    n_pass += ok
    tr = c["spline"](grid)
    pw = np.abs(np.fft.rfft(tr - tr.mean())) ** 2
    hi_frac = pw[freqs > 0.5].sum() / pw[freqs > 0].sum()
    c.update(sd_none=s_none, sd_spline=s_spl, scr=scr, trace_sd=tr.std(), hi_frac=hi_frac)
    print(f"{c['reg']:11s} {c['frac']:4.2f} {c['feats']:9s} {c['mcs']:4d} {'penalized' if c['pen'] else 'lsq':9s} {s_none:8.2f} {s_spl:9.2f} {p5:7.2f} {np.median(scr):7.2f} {str(bool(ok)):>5s} {tr.min():8.1f} {tr.max():8.1f} {tr.std():6.1f} {hi_frac:7.1%} {c['info']['edf']:6.0f}")
print(f"combinations where the spline beats the 5th percentile of the scrambled control: {n_pass} of {len(combos)}")
print("penalties chosen by GCV:", sorted({c['info']['lam'] for c in combos if c['pen']}))
print("The spline is fit to the same centred depths it is evaluated on, so the comparison is partly circular.")

In [ ]:
c = next(c for c in combos if c["reg"] == "a_nonrigid" and c["frac"] == 0.10 and c["feats"] == "x,y,amp" and c["mcs"] == 100 and c["pen"])
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
for ax, (a, b) in zip(axes, [(0, T_END), (200, 230)]):
    m = (c["tt"] >= a) & (c["tt"] < b)
    ax.scatter(c["tt"][m], c["cen"][m], s=0.5, c="tab:blue", rasterized=True)
    g = grid[(grid >= a) & (grid < b)]
    ax.plot(g, c["spline"](g), "r", lw=0.8)
    ax.set_ylim(-300, 300)
    ax.set_xlabel("time (s)"); ax.set_ylabel("centred depth (um)")
    print(f"{a:.0f} to {b:.0f} s: {int(m.sum())} spikes, {int((np.abs(c['cen'][m]) > 300).sum())} outside the y axis")
axes[0].set_title("centred clustered spikes and penalized spline, nonrigid registration, top 10%, x,y,amp, mcs 100")
plt.show()

**Result: reproduced.** In all 32 combinations the pooled within-cluster s.d. after spline correction is below the 5th percentile of the scrambled control (for example 44.30 against 64.66 um with nonrigid registration, top 10%, all three features, min_cluster_size 100, penalized spline). The comparison is partly circular, since the spline is fit to the spikes it is evaluated on. The penalized spline's trace ranges from -208.0 to 350.7 um over the combinations, with s.d. 31.0 to 45.3 um, and 29.4% to 50.4% of its power lies above 0.5 Hz. The least-squares spline takes values of thousands of um (down to -14518.3 um) in stretches without clustered spikes, though not at the spikes it is fit to.

## edf3de-spline-improves-registration

> Correcting the AP-registered spikes with the spline estimate of sub-second displacement gives a
> more stable spike raster and tighter spike localizations than AP DREDge registration alone.

Panel d: mean template correlation of all detected spikes after DREDge, and after DREDge plus the
spline (subtracted from each spike's registered depth at its time), on 1 s bins; 0.1 s bins as a
variant. Panel e: pooled within-cluster sd of corrected depth before and after the spline.

In [ ]:
print(f"{'reg':11s} {'cut':>4s} {'features':9s} {'mcs':>4s} {'spline':9s} {'TC dredge':>9s} {'TC +spl':>8s} {'d up':>5s} {'TC0.1 dr':>8s} {'TC0.1 +s':>8s} {'sd dredge':>9s} {'sd +spl':>8s} {'e down':>6s}")
tc01_cache = {}
n_d = n_e = 0
for c in combos:
    me = reg[c["reg"]]
    yr = yreg[c["reg"]]
    lo, hi = P.probe_extent(me, YLO, YHI)
    base = tc_reg[c["reg"]]
    y2 = yr - c["spline"](sp["t"])
    lo2, hi2 = P.probe_extent(me, YLO, YHI, extra=c["spline"])
    tc2, _ = P.template_correlation(sp["t"], y2, sp["amp"], lo2, hi2, t_max=T_END)
    if c["reg"] not in tc01_cache:
        tc01_cache[c["reg"]] = P.template_correlation(sp["t"], yr, sp["amp"], lo, hi, bin_s=0.1, t_max=T_END)[0]
    tc01, _ = P.template_correlation(sp["t"], y2, sp["amp"], lo2, hi2, bin_s=0.1, t_max=T_END)
    d_up = tc2 > base
    e_down = c["sd_spline"] < c["sd_none"]
    n_d += d_up
    n_e += e_down
    c.update(tc_spline=tc2, tc01_spline=tc01)
    print(f"{c['reg']:11s} {c['frac']:4.2f} {c['feats']:9s} {c['mcs']:4d} {'penalized' if c['pen'] else 'lsq':9s} {base:9.4f} {tc2:8.4f} {str(bool(d_up)):>5s} {tc01_cache[c['reg']]:8.4f} {tc01:8.4f} {c['sd_none']:9.2f} {c['sd_spline']:8.2f} {str(bool(e_down)):>6s}")
print(f"panel d, template correlation higher with the spline: {n_d} of {len(combos)}")
print(f"panel e, pooled within-cluster sd lower with the spline: {n_e} of {len(combos)}")
print("variant (0.1 s bins), template correlation higher with the spline:",
      sum(c["tc01_spline"] > tc01_cache[c["reg"]] for c in combos), "of", len(combos))
for feats in ("x,y,amp", "y,amp"):
    ch = np.array([c["tc_spline"] - tc_reg[c["reg"]] for c in combos if c["feats"] == feats])
    print(f"features {feats}: template correlation change with the spline from {ch.min():+.4f} to {ch.max():+.4f}, higher in {(ch > 0).sum()} of {len(ch)}")
fails = [(c["reg"], c["frac"], c["feats"], c["mcs"], "penalized" if c["pen"] else "lsq") for c in combos if c["tc_spline"] <= tc_reg[c["reg"]]]
print("combinations where it is not higher:", fails)

**Result: partially reproduced.** The pooled within-cluster s.d. is lower after spline correction in all 32 combinations (panel e), but the template correlation of all detected spikes is higher in 22 of 32 (panel d). The 10 combinations where it is not all cluster on registered depth and amplitude without lateral position; with lateral position, depth and amplitude it is higher in all 16, by 0.0362 to 0.0890. One measure improving under every combination and the other not is the criterion's partially reproduced. With 0.1 s bins, a variant outside the outcome, the correlation is higher in 25 of 32.

## Claims on data outside this dandiset

The claims file lists 30 further claims with `data: elsewhere`: human recordings (Dryad and
DANDI:000397), IBL data on AWS, NP Ultra data (DANDI:000957), the macaque insertions of
DANDI:001282, Figshare imposed-motion data, chronic mouse recordings, rat data on G-node and MEArec
simulations. None is attempted here: each needs its own recordings, spike sorting or Kilosort runs,
beyond this run's time. They are recorded as blocked in the attempt.

## Summary

| Claim | Outcome |
|---|---|
| edf3a-ap-dredge-slow-trend | reproduced (assumed: step 1 reading (ii) only) |
| edf3b-hdbscan-clusters | reproduced (assumed: step 1 reading (ii) only) |
| edf3c-spline-subsecond | reproduced (assumed: step 1 reading (ii) only) |
| edf3de-spline-improves-registration | partially reproduced (assumed: step 1 reading (ii) only) |
| 30 claims on data outside the dandiset | blocked, not attempted in this run |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: spikes are detected and localized with SpikeInterface
as in the DREDge repository's demo notebook (reading (ii)), not with the IBL preprocessing and
DARTsort (reading (i)), which did not run in the time available; the paper does not say which
detection it used for this recording. The amplitude cut, the HDBSCAN features and minimum cluster
size, and the spline's penalty are not given in the paper and were run under each reading the
claims file names. The paper gives no measure for these panels; the measures are the claims
file's: the paper's template correlation for the rasters and a pooled within-cluster s.d. for the
localizations, with a scrambled control borrowed from Fig. 2e.

What was found about the dataset: it holds one 600 s Neuropixels 1.0 recording of one macaque,
AP and LF bands, with no trial, stimulus, unit or processed tables, so only the motion analysis of
Extended Data Fig. 3 can be run on it. The session description names static gratings at 8
orientations, where the Methods describe mostly noise stimuli. The dandiset's description cites
"Supp. Fig. 3", the preprint's numbering, for what the paper calls Extended Data Fig. 3. The probe
is described only as "A Neuropixel probe of unknown subtype", and every electrode location is
"unknown", where the description says primary visual cortex. The motion the AP estimate finds is
mostly fast, varying by tens of um within each minute.